# Step 5 - Train a camera-to-command neural network in Google Colab

This notebook trains a small CNN to predict **forward**, **left**, or **right** from a camera image. It uses the existing clean dataset ZIP and saves the trained model to Google Drive. Running this notebook does not connect to or control the robot.

Before starting, upload `dataset_clean_1395.zip` to `MyDrive/Intelligent_Robotics_Design_Project` in Google Drive. The audited clean dataset contains 820 forward, 246 left, and 329 right images (1395 total), each originally 50 x 50 RGB. Four redundant/conflicting files were excluded from the training copy; the raw collection is unchanged.

Open this notebook in Colab and run cells from top to bottom. A GPU runtime is optional for this small model. The notebook uses packages normally available in Colab; no installation cell is needed.


## 1. Configuration

Edit paths and training settings here. The Drive folder holds both the uploaded ZIP and the final model. Keep the image size at 50 x 50 for the agreed camera input format.


In [ ]:
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Intelligent_Robotics_Design_Project"
DATASET_ZIP_NAME = "dataset_clean_1395.zip"
MODEL_FILENAME = "maze_navigation_model.h5"
RANDOM_SEED = 42
IMAGE_SIZE = (50, 50)
BATCH_SIZE = 32
EPOCHS = 30


## 2. Mount Google Drive and import tools

Approve Colab's Drive access prompt. TensorFlow handles image loading and training; scikit-learn supplies stratified splitting, class weights, and evaluation reports. A fixed seed and deterministic TensorFlow operations make repeat runs more reproducible, although results can still vary across software/hardware versions.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import zipfile
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report

tf.keras.utils.set_random_seed(RANDOM_SEED)
tf.config.experimental.enable_op_determinism()
project_dir = Path(DRIVE_PROJECT_DIR)
zip_path = project_dir / DATASET_ZIP_NAME
if not zip_path.is_file():
    raise FileNotFoundError(f"Upload the dataset ZIP here first: {zip_path}")
print(f"Dataset ZIP: {zip_path}")


## 3. Extract the ZIP and verify the class folders

The archive is extracted into `/content/maze_dataset` (temporary Colab storage). Rerunning this cell replaces only that extracted copy. The ZIP can contain the three class folders directly or inside one enclosing folder. Unexpected folders, ambiguous roots, or unsafe archive paths cause a clear error.

**The output class mapping is fixed: forward -> 0, left -> 1, right -> 2. Preserve this mapping with the model; Step 6 will depend on it.**


In [ ]:
CLASS_NAMES = ["forward", "left", "right"]
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}
print("MODEL OUTPUT CLASS MAPPING:")
for name, index in CLASS_TO_INDEX.items():
    print(f"  {name} -> {index}")

extract_dir = Path("/content/maze_dataset")
with zipfile.ZipFile(zip_path) as archive:
    # Check every member before extraction, including absolute/traversal paths.
    for member in archive.infolist():
        destination = (extract_dir / member.filename).resolve()
        if not destination.is_relative_to(extract_dir.resolve()):
            raise ValueError(f"Unsafe ZIP member: {member.filename}")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)
    archive.extractall(extract_dir)

# Unwrap a single enclosing folder, then require exactly the three classes.
dataset_dir = extract_dir
while True:
    entries = list(dataset_dir.iterdir())
    if len(entries) == 1 and entries[0].is_dir() and entries[0].name not in CLASS_NAMES:
        dataset_dir = entries[0]
    else:
        break
if {p.name for p in dataset_dir.iterdir()} != set(CLASS_NAMES):
    raise ValueError(f"Expected exactly {CLASS_NAMES} in {dataset_dir}; found {[p.name for p in dataset_dir.iterdir()]}")
if not all((dataset_dir / name).is_dir() for name in CLASS_NAMES):
    raise ValueError("Every class entry must be a directory.")
print(f"Verified dataset root: {dataset_dir}")


## 4. Validate every image and build the labelled table

Read every JPEG as RGB before splitting. Invalid or unexpected files stop the notebook instead of silently changing the dataset. Print class counts so you can compare them with the clean dataset audit. The table records image paths and integer command labels. Images are resized later by the same preprocessing function used for training and inference.


In [ ]:
records = []
invalid_files = []
original_shapes = set()
for class_name in CLASS_NAMES:
    class_dir = dataset_dir / class_name
    for path in sorted(class_dir.iterdir()):
        if not path.is_file() or path.suffix.lower() not in {".jpg", ".jpeg"}:
            invalid_files.append(f"{path}: expected a JPEG file")
            continue
        try:
            decoded = tf.io.decode_jpeg(tf.io.read_file(str(path)), channels=3)
            shape = tuple(decoded.shape.as_list())
            if len(shape) != 3 or shape[0] <= 0 or shape[1] <= 0 or shape[2] != 3:
                raise ValueError(f"Invalid RGB shape: {shape}")
            original_shapes.add(shape)
            records.append({"path": str(path), "label": CLASS_TO_INDEX[class_name]})
        except Exception as error:
            invalid_files.append(f"{path}: {error}")
if invalid_files:
    raise ValueError("Dataset validation failed:\n" + "\n".join(invalid_files))
images_df = pd.DataFrame(records, columns=["path", "label"])
counts = images_df["label"].value_counts().reindex(range(3), fill_value=0)
print("Validated class counts:")
for index, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {counts[index]}")
print(f"Total: {len(images_df)}")
print(f"Original image shapes: {sorted(original_shapes)}")
if (counts < 7).any():
    raise ValueError("Each class needs enough images for stratified train/validation/test splits.")
print(images_df.head().to_string(index=False))


## 5. Make stratified train, validation, and test splits

Reserve 30% of images, then divide that remainder equally into validation and test sets. This gives approximately 70%/15%/15%, with integer rounding, while retaining each class's proportions using seed 42. Training updates weights; validation guides early stopping; test images are held aside until final evaluation.

These are image-level splits. Nearby frames from the same recording may be visually similar across splits, so test accuracy is not a guarantee of performance on a new route or recording.


In [ ]:
train_df, held_out_df = train_test_split(
    images_df, test_size=0.30, stratify=images_df["label"], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    held_out_df, test_size=0.50, stratify=held_out_df["label"], random_state=RANDOM_SEED
)
for name, frame in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name}: {len(frame)} images ({len(frame) / len(images_df):.1%})")
    print(frame["label"].value_counts().reindex(range(3), fill_value=0).rename(index=dict(enumerate(CLASS_NAMES))).to_string())
assert set(train_df.path).isdisjoint(val_df.path)
assert set(train_df.path).isdisjoint(test_df.path)
assert set(val_df.path).isdisjoint(test_df.path)


## 6. Prepare TensorFlow image pipelines

Decode JPEGs into RGB, resize to 50 x 50, cast to float32, and divide by 255 to scale pixels to [0, 1]. Shuffle only the training set. Batch and prefetch to keep training efficient. Validation and test order stays fixed so predictions match their labels.

There is no horizontal flipping: mirroring an image would change left/right driving semantics.


In [ ]:
def load_image(path):
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    return tf.cast(image, tf.float32) / 255.0

def load_example(path, label):
    return load_image(path), label

def make_dataset(frame, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((
        frame["path"].to_numpy(), frame["label"].to_numpy(dtype=np.int32)
    ))
    dataset = dataset.map(load_example, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)
    if training:
        dataset = dataset.shuffle(len(frame), seed=RANDOM_SEED, reshuffle_each_iteration=True)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)


## 7. Build a modest CNN and handle class imbalance

Three convolution/pooling stages learn visual features. A small dense layer combines those features, dropout helps reduce overfitting, and the final softmax gives three class probabilities in the fixed class order.

Forward images are more common. Balanced class weights are computed **only from training labels** and used during fitting so rarer commands contribute more to training loss. Valid images are retained without fabricating new images.


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(*IMAGE_SIZE, 3)),
    tf.keras.layers.Conv2D(16, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
weights = compute_class_weight(
    class_weight="balanced", classes=np.arange(3), y=train_df["label"].to_numpy()
)
class_weights = {index: float(weight) for index, weight in enumerate(weights)}
print("Training class weights:", {CLASS_NAMES[i]: w for i, w in class_weights.items()})


## 8. Train with early stopping

Train for at most 30 epochs. If validation loss stops improving for five epochs, early stopping ends training and restores the best weights. Validation loss is unweighted; training loss includes class weights and dropout, so their values need not match directly.


In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=[early_stopping],
)


## 9. Plot learning curves

Compare training and validation accuracy/loss. Improving training scores alongside worsening validation scores can indicate overfitting. These plots help explain the model's behaviour in your submission.


In [ ]:
epoch_numbers = range(1, len(history.history["loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, metric, title in zip(axes, ["accuracy", "loss"], ["Accuracy", "Loss"]):
    axis.plot(epoch_numbers, history.history[metric], label="Training")
    axis.plot(epoch_numbers, history.history[f"val_{metric}"], label="Validation")
    axis.set(xlabel="Epoch", ylabel=title, title=f"Training and validation {title.lower()}")
    axis.legend()
    axis.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 10. Evaluate on the untouched test set

Evaluate the restored final model once on held-out test images. The confusion matrix uses true labels as rows and predicted labels as columns. The report gives precision, recall, and F1 for each driving command; these reveal mistakes that overall accuracy can hide in an imbalanced dataset. Do not tune the model using the test results.


In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f} ({test_accuracy:.2%})")
test_probabilities = model.predict(test_ds, verbose=0)
test_predictions = np.argmax(test_probabilities, axis=1)
test_labels = test_df["label"].to_numpy()
confusion = confusion_matrix(test_labels, test_predictions, labels=np.arange(3))
print("Confusion matrix (rows=true, columns=predicted):")
print(pd.DataFrame(confusion, index=CLASS_NAMES, columns=CLASS_NAMES).to_string())
print("Classification report:")
print(classification_report(
    test_labels, test_predictions, labels=np.arange(3),
    target_names=CLASS_NAMES, digits=4, zero_division=0
))


## 11. Save the trained model to Google Drive

Save in the professor's required HDF5 (`.h5`) format. Keras may display a legacy-format warning; HDF5 is intentional for this deliverable. Rerunning this cell overwrites the configured model file with the newly trained model. Check the printed full path and keep this file for submission.


In [ ]:
project_dir.mkdir(parents=True, exist_ok=True)
model_path = project_dir / MODEL_FILENAME
model.save(str(model_path))
if not model_path.is_file() or model_path.stat().st_size == 0:
    raise RuntimeError(f"Model save verification failed: {model_path}")
print(f"Verified saved HDF5 model: {model_path}")
print(f"File size: {model_path.stat().st_size:,} bytes")


## 12. Demonstrate prediction on one test image

Use the same RGB, resize, and scaling steps for a single held-out image. The largest softmax probability selects the predicted command. Probabilities sum to approximately one, but are not a guarantee that a prediction is correct.


In [ ]:
example = test_df.iloc[0]
example_image = load_image(example["path"])
probabilities = model.predict(example_image[None, ...], verbose=0)[0]
predicted_index = int(np.argmax(probabilities))
print(f"True label: {CLASS_NAMES[int(example['label'])]}")
print(f"Predicted label: {CLASS_NAMES[predicted_index]}")
print("Softmax probabilities:")
for name, probability in zip(CLASS_NAMES, probabilities):
    print(f"  {name}: {probability:.6f}")
plt.imshow(example_image.numpy())
plt.title(f"True: {CLASS_NAMES[int(example['label'])]} | Predicted: {CLASS_NAMES[predicted_index]}")
plt.axis("off")
plt.show()


## 13. Preserve the submission outputs

After a successful top-to-bottom Colab run, preserve:

- **`step05_train_neural_network.ipynb`**: save/download the executed notebook with learning curves, test metrics, confusion matrix, classification report, and prediction output visible.
- **`maze_navigation_model.h5`**: the trained neural network saved at `/content/drive/MyDrive/Intelligent_Robotics_Design_Project/maze_navigation_model.h5`.
- **`dataset_clean_1395.zip`**: the separate existing dataset deliverable; this notebook does not create or change it.

Record the output mapping alongside your submission: **forward -> 0, left -> 1, right -> 2**, with RGB inputs resized to 50 x 50 and scaled to [0, 1]. Temporary extracted images in `/content/maze_dataset` need not be preserved. This completes Step 5 training; robot integration belongs to a later step.
